In [9]:
import configparser
import os
import boto3

CREDENTIALS_PATH = "../.aws/credentials"

AWS_REGION = "us-east-1"

aws_config = configparser.ConfigParser()
aws_config.read(CREDENTIALS_PATH)

PROFILE = aws_config.sections()[0]

AWS_ACCESS_KEY = aws_config[PROFILE].get("aws_access_key_id")
AWS_SECRET_KEY = aws_config[PROFILE].get("aws_secret_access_key")
AWS_SESSION_TOKEN = aws_config[PROFILE].get("aws_session_token", None)

In [10]:
def crear_cliente(servicio):
    return boto3.client(
        servicio,
        aws_access_key_id=AWS_ACCESS_KEY,
        aws_secret_access_key=AWS_SECRET_KEY,
        aws_session_token=AWS_SESSION_TOKEN,
        region_name=AWS_REGION
    )

Probar que las credenciales funcionan:

In [12]:
sts = crear_cliente("sts")
print(sts.get_caller_identity())

{'UserId': 'AROAXF33W5AWU2HEZBHCE:aitaguduq@alu.edu.gva.es', 'Account': '493643819053', 'Arn': 'arn:aws:sts::493643819053:assumed-role/AWSReservedSSO_AWSAdministratorAccess_e1d3988c982ae3e1/aitaguduq@alu.edu.gva.es', 'ResponseMetadata': {'RequestId': 'cec95028-0d34-43e9-a4fe-fe0752aab1e6', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'cec95028-0d34-43e9-a4fe-fe0752aab1e6', 'x-amz-sts-extended-request-id': 'MTp1cy1lYXN0LTE6UzoxNzc3NDAxOTI1MzExOlI6NnFDOWhLTWM=', 'content-type': 'text/xml', 'content-length': '509', 'date': 'Tue, 28 Apr 2026 18:45:25 GMT'}, 'RetryAttempts': 0}}


Comprobar que se reconocen correctamente las matriculas:

In [17]:
import boto3
import re

rekognition = crear_cliente("rekognition")

def detectar_matriculas(image_path):
    with open(image_path, 'rb') as image_file:
        image_bytes = image_file.read()

    response = rekognition.detect_text(
        Image={'Bytes': image_bytes}
    )

    textos = response['TextDetections']

    resultados = []

    for t in textos:
        if t["Type"] == "LINE":
            texto = t["DetectedText"].upper().replace(" ", "").replace("-", "")
            confianza = t["Confidence"]

            print(f"{texto} - {confianza}")

            resultados.append(texto)

    return resultados


In [19]:
imagen = "../assets/coche.jpg"
matriculas = detectar_matriculas(imagen)

WIOC748 - 81.10054016113281


In [ ]:
dynamodb = boto3.resource("dynamodb", region_name=REGION)
client = crear_cliente("dynamodb")


def crear_tabla_si_no_existe():
    try:
        client.describe_table(TableName=TABLE_NAME)
        print("✔ La tabla ya existe")
    except client.exceptions.ResourceNotFoundException:
        print("📦 Creando tabla...")

        client.create_table(
            TableName=TABLE_NAME,
            KeySchema=[
                {"AttributeName": "matricula", "KeyType": "HASH"}
            ],
            AttributeDefinitions=[
                {"AttributeName": "matricula", "AttributeType": "S"}
            ],
            BillingMode="PAY_PER_REQUEST"
        )

        # esperar a que esté activa
        waiter = client.get_waiter("table_exists")
        waiter.wait(TableName=TABLE_NAME)

        print("✔ Tabla creada correctamente")


# 2. Guardar matrícula
def guardar_matricula(matricula):
    table = dynamodb.Table(TABLE_NAME)

    table.put_item(
        Item={
            "matricula": matricula
        }
    )

    print(f"✔ Guardada: {matricula}")